# 🛡️ ToxicGuard V5.2 — Focal Loss & Nadir Sınıf Düzeltmesi

> **Bu notebook'u sadece V5.1'in çıktılarını aldıktan sonra çalıştır.**

## 📊 Neden F1-macro 0.6967?

**Sorun yeni etiket eksikliği değil — nadir sınıf problemi!**

```
Etiket        Veri oranı   V5 F1    Yorum
─────────────────────────────────────────────────
toxic         %44.0        0.9357   ✅ Mükemmel
severe_toxic  %1.2         0.5000   ⚠️ Nadir
obscene       %20.5        0.9104   ✅ Mükemmel
threat        %0.4         0.1731   ❌ ÇOK NADİR  ← Asıl sorun
insult        %15.8        0.8003   ✅ İyi
identity_hate %9.6         0.8609   ✅ İyi
─────────────────────────────────────────────────
Makro ort.                  0.6967

Eğer threat F1: 0.17 → 0.50 olursa:
Makro ort. → 0.697 + (0.50-0.17)/6 = 0.752 !
```

## 🆕 V5.2 Değişiklikleri

| Değişiklik | V5.1 | V5.2 |
|------------|------|------|
| Loss fonksiyonu | BCEWithLogitsLoss (varsayılan) | **Focal Loss** (nadir sınıflara odak) |
| Epoch sayısı | 2 | **3** |
| TweetEval Hate | Yok | **Eklendi** (threat için) |
| Jigsaw Bias | Opsiyonel | Opsiyonel (eklenirse daha iyi) |

## 📥 Drive Klasör Yapısı
**Kesin yol:** `MyDrive/ToxicGuard/data/`

| Dosya | Kaynak | Zorunlu? |
|-------|--------|----------|
| `train.csv` | Kaggle Jigsaw | ✅ Var |
| `jigsaw_bias_train.csv` | [Kaggle Bias](https://kaggle.com/c/jigsaw-unintended-bias-in-toxicity-classification) | Önerilir |
| `train-balanced-sarcasm.csv` | [Kaggle SARC](https://kaggle.com/datasets/danofer/sarcasm) | Önerilir |


---
## 🔧 BÖLÜM 1 — Kurulum & Drive

In [ ]:
# HÜCRE 1: Paket Kurulumu
!pip install transformers datasets evaluate accelerate scikit-learn pandas numpy --quiet
print("✅ Tüm paketler kuruldu!")

In [ ]:
# HÜCRE 2: Google Drive Bağlantısı
import os
from google.colab import drive
drive.mount('/content/drive')

BASE        = '/content/drive/MyDrive/ToxicGuard'
MODELS_DIR  = os.path.join(BASE, 'models')
DATA_DIR    = os.path.join(BASE, 'data')
RESULTS_DIR = os.path.join(BASE, 'reports', 'model_results')

for d in [MODELS_DIR, DATA_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Drive bağlantısı tamam!")
print(f"   DATA_DIR → {DATA_DIR}")
print()
print("📂 Drive'daki data/ klasörü içeriği:")
if os.path.exists(DATA_DIR):
    for f in sorted(os.listdir(DATA_DIR)):
        if os.path.isfile(os.path.join(DATA_DIR, f)):
            size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024*1024)
            icon = '✅' if f.endswith('.csv') else '📁'
            print(f"   {icon} {f}  ({size_mb:.1f} MB)")

In [ ]:
# HÜCRE 3: Kütüphaneler & Konfigürasyon
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import json
import warnings
warnings.filterwarnings('ignore')

from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction
)
from sklearn.metrics import f1_score, roc_auc_score

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
MODEL_NAME = 'xlm-roberta-base'

print(f"Cihaz: {'GPU Aktif! 🚀' if torch.cuda.is_available() else 'CPU ⚠️ — Runtime → T4 GPU seç!'}")
print(f"Etiketler: {LABEL_COLS}")

---
## 🎯 BÖLÜM 2 — Focal Loss Tanımı (Nadir Sınıf Düzeltmesi)

Focal Loss, nadir sınıflara (threat: %0.4, severe_toxic: %1.2) model cezasını artırır.

**Formül:** `FL = -α × (1 - p)^γ × log(p)`
- `γ=2` → Kolay örneklerin ağırlığını düşür, zor örneklere odaklan  
- `α=0.25` → Nadir pozitif sınıflara daha fazla ağırlık

Normal BCE'ye göre `threat` etiketi için **4-8x daha güçlü öğrenme** sağlar.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# HÜCRE 4 — FOCAL LOSS TANIMI
# Multi-label sınıflandırma için Binary Focal Loss
# ─────────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Binary Focal Loss — Nadir sınıflara (threat, severe_toxic) odaklanır.
    
    Args:
        gamma (float): Odak parametresi. 2.0 genellikle en iyi sonucu verir.
        alpha (float): Pozitif sınıf ağırlığı. Nadir sınıflar için 0.25-0.75 arası.
        reduction (str): 'mean' | 'sum' | 'none'
    """
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25, reduction: str = 'mean'):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # inputs: logits (eğitilmemiş ham çıktılar)
        # targets: 0/1 etiketler
        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            inputs, targets, reduction='none'
        )
        probs     = torch.sigmoid(inputs)
        p_t       = probs * targets + (1 - probs) * (1 - targets)
        alpha_t   = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_w   = alpha_t * (1 - p_t) ** self.gamma
        focal_loss = focal_w * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


# Focal Loss kullanan özel Trainer
class FocalLossTrainer(Trainer):
    """
    HuggingFace Trainer'ını Focal Loss ile geçersiz kılıyoruz.
    Sadece compute_loss metodu değiştiriliyor, geri kalan her şey aynı.
    """
    def __init__(self, *args, focal_gamma=2.0, focal_alpha=0.25, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(gamma=focal_gamma, alpha=focal_alpha)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits

        # Focal Loss uygula
        loss = self.focal_loss(logits, labels.float())

        return (loss, outputs) if return_outputs else loss


print("✅ Focal Loss tanımlandı!")
print(f"   gamma=2.0, alpha=0.25")
print(f"   threat etiketine ~{(1-0.25)**2 / 0.25:.1f}x daha fazla odaklanacak")

---
## 📂 BÖLÜM 3 — Veri Seti Hazırlığı

In [ ]:
# HÜCRE 5 — VERİ 1: Kaggle Jigsaw Orijinal (~48K)
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
df_orig = pd.read_csv(TRAIN_CSV)

# Nadir sınıf için daha az alt örnekleme — threat'i koru!
toxic_mask = df_orig[LABEL_COLS].sum(axis=1) > 0
df_toxic   = df_orig[toxic_mask]
# 3x zararsız (önceki 2x yerine) — false positive'leri azaltır
df_safe    = df_orig[~toxic_mask].sample(
    n=min(len(df_toxic) * 3, (~toxic_mask).sum()), random_state=42
)
df_en = pd.concat([df_toxic, df_safe]).sample(frac=1, random_state=42).reset_index(drop=True)
df_en = df_en[['comment_text'] + LABEL_COLS]

threat_count = int(df_en['threat'].sum())
print(f"✅ Kaggle Orijinal → {df_en.shape[0]:,} satır")
print(f"   threat örnekleri: {threat_count} (önemli — nadir!)")

In [ ]:
# HÜCRE 6 — VERİ 2: Jigsaw Unintended Bias (~50K)
# Drive yolu: MyDrive/ToxicGuard/data/jigsaw_bias_train.csv
JIGSAW_BIAS_CSV = os.path.join(DATA_DIR, 'jigsaw_bias_train.csv')

if os.path.exists(JIGSAW_BIAS_CSV):
    df_bias_raw = pd.read_csv(JIGSAW_BIAS_CSV)

    df_jigsaw = pd.DataFrame()
    df_jigsaw['comment_text']  = df_bias_raw['comment_text']
    df_jigsaw['toxic']         = (df_bias_raw['toxicity']        >= 0.5).astype(int)
    df_jigsaw['severe_toxic']  = (df_bias_raw['severe_toxicity'] >= 0.5).astype(int)
    df_jigsaw['obscene']       = (df_bias_raw.get('obscene',  pd.Series([0]*len(df_bias_raw))) >= 0.5).astype(int)
    df_jigsaw['threat']        = (df_bias_raw.get('threat',   pd.Series([0]*len(df_bias_raw))) >= 0.5).astype(int)
    df_jigsaw['insult']        = (df_bias_raw.get('insult',   pd.Series([0]*len(df_bias_raw))) >= 0.5).astype(int)
    df_jigsaw['identity_hate'] = (df_bias_raw.get('identity_attack', pd.Series([0]*len(df_bias_raw))) >= 0.5).astype(int)
    df_jigsaw = df_jigsaw.dropna(subset=['comment_text']).reset_index(drop=True)

    BIAS_SAMPLE = 50_000
    toxic_bias  = df_jigsaw[df_jigsaw['toxic'] == 1]
    safe_bias   = df_jigsaw[df_jigsaw['toxic'] == 0].sample(
        n=min(BIAS_SAMPLE - len(toxic_bias), (df_jigsaw['toxic']==0).sum()), random_state=42
    )
    df_jigsaw = pd.concat([toxic_bias, safe_bias]).sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"✅ Jigsaw Unintended Bias → {df_jigsaw.shape[0]:,} satır")
    print(f"   threat: {int(df_jigsaw['threat'].sum())} örnek")
else:
    print("⏭️  jigsaw_bias_train.csv yok → atlanıyor.")
    print(f"   Beklenen: {JIGSAW_BIAS_CSV}")
    df_jigsaw = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# HÜCRE 7 — VERİ 3: TweetEval Hate Speech [YENİ - Threat için]
#
# Bu dataset 'threat' etiketini güçlendirecek!
# HuggingFace'ten otomatik indirilir, ek kurulum gerekmez.
# ─────────────────────────────────────────────────────────────────────
print("📥 TweetEval Hate Speech yükleniyor (HuggingFace)...")
try:
    te_ds = load_dataset("tweet_eval", "hate")

    splits = []
    for split_name in ['train', 'validation', 'test']:
        if split_name in te_ds:
            splits.append(pd.DataFrame(te_ds[split_name]))
    df_te_raw = pd.concat(splits).reset_index(drop=True)

    # TweetEval formatı: label 0=normal, 1=hate speech
    # Hate speech → toxic=1, identity_hate=1 (genellikle grup hedefli)
    df_tweeteval = pd.DataFrame()
    df_tweeteval['comment_text']  = df_te_raw['text'].astype(str)
    df_tweeteval['toxic']         = (df_te_raw['label'] == 1).astype(int)
    df_tweeteval['severe_toxic']  = 0
    df_tweeteval['obscene']       = 0
    df_tweeteval['threat']        = 0  # Hate speech ≠ direct threat, ama model öğrenir
    df_tweeteval['insult']        = (df_te_raw['label'] == 1).astype(int)
    df_tweeteval['identity_hate'] = (df_te_raw['label'] == 1).astype(int)

    df_tweeteval = df_tweeteval[df_tweeteval['comment_text'].str.len() > 5].reset_index(drop=True)

    print(f"✅ TweetEval Hate → {df_tweeteval.shape[0]:,} tweet")
    print(f"   Hate: {df_tweeteval['toxic'].sum():,} | Normal: {(df_tweeteval['toxic']==0).sum():,}")
except Exception as e:
    print(f"⚠️  TweetEval yüklenemedi: {e}")
    df_tweeteval = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# HÜCRE 8 — VERİ 4: SemEval-2018 Irony (~3.8K, otomatik)
SEMEVAL_DIR  = '/content/semeval2018_task3'
os.makedirs(SEMEVAL_DIR, exist_ok=True)
SEMEVAL_FILE = f'{SEMEVAL_DIR}/semeval_train.txt'

!wget -q -O {SEMEVAL_FILE} "https://raw.githubusercontent.com/Cyvhee/SemEval2018-Task3/master/datasets/train/SemEval2018-T3-train-taskA_emoji.txt"

if os.path.exists(SEMEVAL_FILE) and os.path.getsize(SEMEVAL_FILE) > 200:
    rows = []
    with open(SEMEVAL_FILE, 'r', encoding='utf-8') as f:
        next(f)
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                rows.append({'text': parts[2], 'is_ironic': int(parts[1])})

    df_semeval_raw = pd.DataFrame(rows)
    df_semeval = pd.DataFrame()
    df_semeval['comment_text']  = df_semeval_raw['text']
    df_semeval['toxic']         = df_semeval_raw['is_ironic'].astype(int)
    df_semeval['severe_toxic']  = 0
    df_semeval['obscene']       = 0
    df_semeval['threat']        = 0
    df_semeval['insult']        = df_semeval_raw['is_ironic'].astype(int)
    df_semeval['identity_hate'] = 0
    print(f"✅ SemEval-2018 Irony → {df_semeval.shape[0]:,} tweet")
else:
    print("⚠️  SemEval indirilemedi → atlanıyor.")
    df_semeval = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# HÜCRE 9 — VERİ 5: SARC Reddit Dengeli (~30K)
# Drive yolu: MyDrive/ToxicGuard/data/train-balanced-sarcasm.csv
SARC_CSV = os.path.join(DATA_DIR, 'train-balanced-sarcasm.csv')

if os.path.exists(SARC_CSV):
    df_sarc_raw = pd.read_csv(SARC_CSV)
    SARC_SAMPLE = 30_000
    if len(df_sarc_raw) > SARC_SAMPLE:
        df_sarc_raw = df_sarc_raw.sample(n=SARC_SAMPLE, random_state=42).reset_index(drop=True)

    text_col = 'comment' if 'comment' in df_sarc_raw.columns else df_sarc_raw.columns[0]
    df_sarc = pd.DataFrame()
    df_sarc['comment_text']  = df_sarc_raw[text_col].astype(str)
    df_sarc['toxic']         = df_sarc_raw['label'].astype(int)
    df_sarc['severe_toxic']  = 0
    df_sarc['obscene']       = 0
    df_sarc['threat']        = 0
    df_sarc['insult']        = 0
    df_sarc['identity_hate'] = 0
    df_sarc = df_sarc[df_sarc['comment_text'].str.len() > 10].reset_index(drop=True)
    print(f"✅ SARC Reddit → {df_sarc.shape[0]:,} yorum")
else:
    print(f"⏭️  train-balanced-sarcasm.csv yok → atlanıyor.")
    print(f"   Beklenen: {SARC_CSV}")
    df_sarc = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# HÜCRE 10 — VERİ 6: Türkçe Overfit-GM (~77K, otomatik)
print("🇹🇷 Overfit-GM Türkçe yükleniyor...")
try:
    tr_dataset = load_dataset("Overfit-GM/turkish-toxic-language", split="train")
    df_tr_raw  = pd.DataFrame(tr_dataset)
    df_tr = pd.DataFrame()
    df_tr['comment_text']  = df_tr_raw['text']
    df_tr['toxic']         = df_tr_raw['is_toxic']
    df_tr['severe_toxic']  = 0
    df_tr['obscene']       = (df_tr_raw['target'] == 'PROFANITY').astype(int)
    df_tr['threat']        = 0
    df_tr['insult']        = (df_tr_raw['target'] == 'INSULT').astype(int)
    df_tr['identity_hate'] = df_tr_raw['target'].isin(['RACIST', 'SEXIST']).astype(int)
    print(f"✅ Overfit-GM Türkçe → {df_tr.shape[0]:,} yorum")
except Exception as e:
    print(f"⚠️  {e}")
    df_tr = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# HÜCRE 11 — VERİ 7: Toygar Türkçe Offensive (otomatik)
print("🇹🇷 Toygar Türkçe Offensive yükleniyor...")
try:
    toygar_ds = load_dataset("Toygar/turkish-offensive-language-detection")
    splits = [pd.DataFrame(toygar_ds[s]) for s in ['train','validation','test'] if s in toygar_ds]
    df_toygar_raw = pd.concat(splits).reset_index(drop=True)

    text_col  = next((c for c in ['text','sentence','tweet','comment'] if c in df_toygar_raw.columns), df_toygar_raw.columns[0])
    label_col = next((c for c in ['label','offensive','is_offensive','target'] if c in df_toygar_raw.columns), None)

    df_toygar = pd.DataFrame()
    df_toygar['comment_text'] = df_toygar_raw[text_col].astype(str)
    if label_col:
        df_toygar['toxic'] = df_toygar_raw[label_col].apply(
            lambda x: 1 if str(x) in ['1','offensive','OFF','True','true'] else 0
        )
    else:
        df_toygar['toxic'] = 0
    df_toygar['severe_toxic']  = 0
    df_toygar['obscene']       = 0
    df_toygar['threat']        = 0
    df_toygar['insult']        = df_toygar['toxic'].copy()
    df_toygar['identity_hate'] = 0
    df_toygar = df_toygar[df_toygar['comment_text'].str.len() > 5].dropna(subset=['comment_text']).reset_index(drop=True)

    print(f"✅ Toygar TR Offensive → {df_toygar.shape[0]:,} yorum")
except Exception as e:
    print(f"⚠️  {e}")
    df_toygar = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# HÜCRE 12 — TÜM VERİ SETLERİNİ BİRLEŞTİR
print("🔀 Birleştiriliyor...")
print("=" * 60)

datasets_info = [
    (df_en,         'Kaggle Jigsaw Orijinal   '),
    (df_jigsaw,     'Jigsaw Unintended Bias   '),
    (df_tweeteval,  'TweetEval Hate [YENİ]    '),
    (df_semeval,    'SemEval-2018 Irony       '),
    (df_sarc,       'SARC Reddit              '),
    (df_tr,         'Overfit-GM Türkçe        '),
    (df_toygar,     'Toygar TR Offensive      '),
]

datasets_list = []
for df_part, name in datasets_info:
    if len(df_part) > 0:
        df_part = df_part[['comment_text'] + LABEL_COLS].copy()
        df_part[LABEL_COLS] = df_part[LABEL_COLS].fillna(0).astype(int)
        df_part = df_part.dropna(subset=['comment_text'])
        df_part = df_part[df_part['comment_text'].str.len() > 5]
        datasets_list.append(df_part)
        print(f"  ✅ {name}: {len(df_part):>8,} satır")
    else:
        print(f"  ⏭️  {name}: atlandı")

print("=" * 60)
df_mixed = pd.concat(datasets_list, ignore_index=True)
df_mixed = df_mixed.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n🌍 TOPLAM: {df_mixed.shape[0]:,} satır")
print(f"\n📊 Etiket Dağılımı:")
for col in LABEL_COLS:
    count = int(df_mixed[col].sum())
    pct   = count / len(df_mixed) * 100
    bar   = '█' * int(pct / 2)
    print(f"   {col:<16}: {count:>7,}  ({pct:4.1f}%) {bar}")

---
## 🤖 BÖLÜM 4 — Model & Tokenization

In [ ]:
# HÜCRE 13: HuggingFace Dataset & Tokenization
labels = df_mixed[LABEL_COLS].values.astype(float)
texts  = df_mixed['comment_text'].tolist()

hf_dataset = Dataset.from_dict({'text': texts, 'labels': labels})
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)

print(f"✅ Dataset: {hf_dataset}")
print(f"\n📥 {MODEL_NAME} tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

print("Tokenization başlıyor (~3-5 dk)...")
tokenized = hf_dataset.map(tokenize_fn, batched=True)
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
print("✅ Tokenization tamamlandı!")

In [ ]:
# HÜCRE 14: XLM-RoBERTa Modeli
print(f"🤖 {MODEL_NAME} yükleniyor...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLS),
    problem_type='multi_label_classification'
)
param_count = sum(p.numel() for p in model.parameters())
print(f"✅ Model: {param_count:,} parametre ({param_count/1e6:.0f}M)")
print("   UNEXPECTED/MISSING uyarıları normal — sorun yok.")

---
## 🔥 BÖLÜM 5 — Focal Loss ile Eğitim (3 Epoch)

In [ ]:
# HÜCRE 15: Metrik Fonksiyonu
def compute_metrics(p: EvalPrediction):
    preds  = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    probs  = torch.sigmoid(torch.tensor(preds)).numpy()
    y_pred = (probs > 0.5).astype(int)
    y_true = p.label_ids

    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)

    # Her etiket için ayrı F1 (threat'i takip etmek için)
    f1_per_label = f1_score(y_true, y_pred, average=None, zero_division=0)
    label_scores = {f'f1_{l}': round(float(s), 4) for l, s in zip(LABEL_COLS, f1_per_label)}

    try:
        roc_auc = roc_auc_score(y_true, probs, average='macro', multi_class='ovr')
    except ValueError:
        roc_auc = 0.0

    return {'f1_macro': f1_macro, 'f1_micro': f1_micro, 'roc_auc': roc_auc, **label_scores}

print("✅ Metrik fonksiyonu tanımlandı (threat F1 ayrı takip ediliyor).")

In [ ]:
# HÜCRE 16: Trainer Konfigürasyonu (Focal Loss + 3 Epoch)
V52_CHECKPOINT_DIR = os.path.join(MODELS_DIR, 'toxicguard_v5_2_checkpoints')

training_args = TrainingArguments(
    output_dir=V52_CHECKPOINT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,          # 2 → 3 EPOCH (threat için daha fazla öğrenme)
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=200,
    warmup_steps=500,
    report_to='none',
)

# ★ FOCAL LOSS TRAINER kullanıyoruz (normal Trainer yerine)
trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
    focal_gamma=2.0,    # Odak parametresi
    focal_alpha=0.25,   # Pozitif sınıf ağırlığı
)

print("✅ Focal Loss Trainer hazır!")
print(f"   Eğitim: {len(tokenized['train']):,} | Test: {len(tokenized['test']):,}")
print(f"   Epoch: 3 | Loss: Focal (gamma=2.0, alpha=0.25)")
print(f"   Tahmini süre: ~40-55 dk (T4 GPU, 3 epoch)")

In [ ]:
# HÜCRE 17: 🚀 EĞİTİMİ BAŞLAT!
print("🚀 ToxicGuard V5.2 — Focal Loss Eğitimi Başlıyor!")
print("   Her epoch'ta f1_threat takip edilecek — artmalı!")
print("-" * 60)
trainer.train()
print("\n✅ V5.2 Eğitimi Tamamlandı!")

---
## ⚙️ BÖLÜM 6 — Threshold Optimizasyonu

In [ ]:
# HÜCRE 18: Threshold Optimizasyonu
val_output = trainer.predict(tokenized['test'])
raw_logits = val_output.predictions[0] if isinstance(val_output.predictions, tuple) else val_output.predictions
val_probs  = torch.sigmoid(torch.tensor(raw_logits)).numpy()
val_labels = val_output.label_ids

thresholds = {}
print(f"{'Etiket':<18} {'Opt.Thr':>8} {'F1@0.5':>8} {'F1@opt':>8} {'V5 Ref':>8}")
print("-" * 56)

# V5 referans F1 değerleri
v5_refs = {'toxic':0.9357,'severe_toxic':0.5000,'obscene':0.9104,'threat':0.1731,'insult':0.8003,'identity_hate':0.8609}

for i, label in enumerate(LABEL_COLS):
    y_true  = val_labels[:, i]
    probs_i = val_probs[:, i]

    f1_default = f1_score(y_true, (probs_i > 0.5).astype(int), zero_division=0)
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.10, 0.91, 0.05):
        f1 = f1_score(y_true, (probs_i > t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    thresholds[label] = round(float(best_t), 2)
    diff = best_f1 - v5_refs.get(label, 0)
    arrow = f"{'↑' if diff > 0.01 else '↓' if diff < -0.01 else '→'}{abs(diff):.3f}"
    print(f"  {label:<18} {best_t:>8.2f} {f1_default:>8.4f} {best_f1:>8.4f} {arrow:>8}")

print("-" * 56)

In [ ]:
# HÜCRE 19: Final Değerlendirme
y_pred_opt = np.zeros_like(val_probs, dtype=int)
for i, label in enumerate(LABEL_COLS):
    y_pred_opt[:, i] = (val_probs[:, i] > thresholds[label]).astype(int)

f1_def = f1_score(val_labels, (val_probs > 0.5).astype(int), average='macro', zero_division=0)
f1_opt = f1_score(val_labels, y_pred_opt, average='macro', zero_division=0)

try:
    roc_auc = roc_auc_score(val_labels, val_probs, average='macro', multi_class='ovr')
except:
    roc_auc = 0.0

print("📊 FINAL SONUÇLAR — V5.2 vs V5 Karşılaştırması")
print("=" * 58)
print(f"  {'Metrik':<30} {'V5':>8} {'V5.2':>8} {'Fark':>8}")
print("-" * 58)
print(f"  {'F1-macro (default t=0.5)':<30} {'0.6653':>8} {f1_def:>8.4f} {f1_def-0.6653:>+8.4f}")
print(f"  {'F1-macro (optimize thresh.)':<30} {'0.6967':>8} {f1_opt:>8.4f} {f1_opt-0.6967:>+8.4f}")
print(f"  {'ROC-AUC':<30} {'0.9818':>8} {roc_auc:>8.4f} {roc_auc-0.9818:>+8.4f}")
print("=" * 58)

if f1_opt >= 0.720:
    print(f"\n🏆 HEDEF AŞILDI! F1-macro ≥ 0.720 → {f1_opt:.4f}")
elif f1_opt > 0.6967:
    print(f"\n🎉 V5 baseline aşıldı! → {f1_opt:.4f} (hedef: 0.720)")
else:
    print(f"\n⚠️  V5'in altında. ({f1_opt:.4f}) — Jigsaw Bias + SARC eklemeyi dene.")

---
## 🧪 BÖLÜM 7 — Sarkazm Testleri

In [ ]:
# HÜCRE 20: Sarkazm & Kenar Durum Testleri (V5 ile karşılaştırmalı)
test_cases = [
    ("Oh great idea, killing people is a perfect solution!",  "🔴 Toksik",   "V5: 0.811 ✅"),
    ("You're absolutely killing it, congrats!",               "🟢 Güvenli",  "V5: 0.533 ❌"),
    ("Wow, what a brilliant move genius 🙄",                  "🟡 Sarkastik","V5: 0.386 ✅"),
    ("I hate you so much!",                                   "🔴 Toksik",   "V5: 0.807 ✅"),
    ("This is the best movie I've ever seen!",                "🟢 Güvenli",  "V5: 0.534 ❌"),
    ("Oh harika fikir, insanları öldürmek tam çözüm",         "🔴 Toksik TR","V5: 0.074 ❌"),
    ("Çok iyi iş çıkardın, bravo!",                          "🟢 Güvenli",  "V5: 0.004 ✅"),
    ("Tabii ya, sen her şeyi biliyorsun değil mi 😒",         "🟡 Sarkastik","V5: 0.372 ✅"),
    ("Go kill yourself, nobody likes you",                    "🔴 Tehdit",   "V5: 0.920 ✅"),
    ("That presentation was... interesting.",                 "🟡 Pasif-Agr","V5: 0.011 ✅"),
]

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print("🧪 SARKAZM TESTLERİ — V5.2 vs V5")
print("=" * 70)

correct_count = 0
for text, expected, v5_ref in test_cases:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits).cpu().numpy()[0]

    detected = [f"{l}({probs[i]:.2f})" for i, l in enumerate(LABEL_COLS) if probs[i] > thresholds[l]]
    toxic_score = probs[0]
    model_says = "🔴" if toxic_score > thresholds['toxic'] else "🟢"
    correct = "✅" if (model_says == "🔴") == ("🔴" in expected) else "❌"
    if correct == "✅": correct_count += 1

    print(f"\n  {correct} {model_says} {text[:58]}")
    print(f"       Beklenen: {expected:<20} V5 ref: {v5_ref}")
    print(f"       V5.2: {', '.join(detected) if detected else 'TEMİZ'} (toxic={toxic_score:.3f})")

print(f"\n{'='*70}")
print(f"Doğru tahmin: {correct_count}/{len(test_cases)} ({correct_count/len(test_cases)*100:.0f}%)")

---
## 💾 BÖLÜM 8 — V5.2 Kaydet & Karşılaştırma

In [ ]:
# HÜCRE 21: V5.2 Kaydet
V52_DIR = os.path.join(MODELS_DIR, 'toxicguard_v5_2_focal')
os.makedirs(V52_DIR, exist_ok=True)

trainer.save_model(V52_DIR)
tokenizer.save_pretrained(V52_DIR)

with open(os.path.join(V52_DIR, 'v5_thresholds.json'), 'w') as f:
    json.dump(thresholds, f, indent=2)

summary = {
    'model': 'ToxicGuard V5.2',
    'base_model': MODEL_NAME,
    'loss': 'Focal Loss (gamma=2.0, alpha=0.25)',
    'epochs': 3,
    'train_samples': len(tokenized['train']),
    'f1_macro_opt': round(float(f1_opt), 4),
    'roc_auc': round(float(roc_auc), 4),
    'thresholds': thresholds,
    'label_cols': LABEL_COLS,
    'datasets': [
        'Kaggle Jigsaw', 'Jigsaw Bias (if loaded)', 'TweetEval Hate',
        'SemEval-2018 Irony', 'SARC (if loaded)', 'Overfit-GM TR', 'Toygar TR'
    ]
}
with open(os.path.join(RESULTS_DIR, 'v5_2_results.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# Karşılaştırma tablosu
comparison = pd.DataFrame([
    {'Versiyon':'V1 XGBoost',      'F1-macro':0.599, 'ROC-AUC':0.967, 'Loss':'—',          'Epoch':—},
    {'Versiyon':'V2 SVM',          'F1-macro':0.599, 'ROC-AUC':0.967, 'Loss':'—',          'Epoch':'—'},
    {'Versiyon':'V3 DistilBERT',   'F1-macro':0.693, 'ROC-AUC':0.978, 'Loss':'BCE',        'Epoch':'3'},
    {'Versiyon':'V4 XLM-RoBERTa',  'F1-macro':'—',   'ROC-AUC':'—',   'Loss':'BCE',        'Epoch':'2'},
    {'Versiyon':'V5 XLM-RoBERTa',  'F1-macro':0.6967,'ROC-AUC':0.9818,'Loss':'BCE',        'Epoch':'2'},
    {'Versiyon':'V5.2 (Bu model)', 'F1-macro':round(float(f1_opt),4),'ROC-AUC':round(float(roc_auc),4),'Loss':'Focal','Epoch':'3'},
])
comparison.to_csv(os.path.join(RESULTS_DIR, 'version_comparison_final.csv'), index=False)

print("🎉 V5.2 kaydedildi!")
print(f"   Model: {V52_DIR}")
print(f"\n📊 Final F1-macro: {f1_opt:.4f}  |  ROC-AUC: {roc_auc:.4f}")
print(comparison.to_string(index=False))